# Error Analysis

Analyse inference results to understand where the model is making mistakes,
then use those insights to manually craft a `category_distribution` for the
next `build_quota_dataset` run.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

RESULTS_DIR = Path("../results/inference_results")

## Load results

Pick one or more results files to analyse.

In [ ]:
# List available result files
result_files = sorted(RESULTS_DIR.rglob("*.json"))
for f in result_files:
    print(f.relative_to(RESULTS_DIR))

In [ ]:
# Edit to select the file(s) you want to analyse
selected_files = [RESULTS_DIR / Path("valid/gemma-4-E2B-20260428-093044.json")]


def load_results(path: Path) -> pd.DataFrame:
    data = json.loads(path.read_text())
    records = list(data.values()) if isinstance(data, dict) else data
    return pd.DataFrame(records)


frames = []
for f in selected_files:
    df = load_results(f)
    df["results_file"] = f.name
    frames.append(df)

frame = pd.concat(frames, ignore_index=True)
print(f"{len(frame):,} records from {len(frames)} file(s)")
frame.head(3)

## Accuracy overview

In [ ]:
frame["correct"] = (
    frame["model_response"].str.strip().str.upper()
    == frame["correct_answer"].str.strip().str.upper()
)

overall_acc = frame["correct"].mean()
print(
    f"Overall accuracy: {overall_acc:.1%}  ({frame['correct'].sum():,} / {len(frame):,})"
)

## Accuracy by category

In [ ]:
by_category = (
    frame.groupby("category")["correct"]
    .agg(total="count", correct="sum")
    .assign(
        accuracy=lambda d: d["correct"] / d["total"],
        error_rate=lambda d: 1 - d["correct"] / d["total"],
    )
    .sort_values("error_rate", ascending=False)
)
by_category

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
by_category["error_rate"].plot.barh(ax=ax, color="steelblue")
ax.axvline(1 - overall_acc, color="red", linestyle="--", label="overall error rate")
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.set_xlabel("Error rate")
ax.set_title("Error rate by category")
ax.legend()
plt.tight_layout()
plt.show()

## Accuracy by subcategory (top errors)

In [ ]:
TOP_N = 20

by_subcategory = (
    frame.groupby(["category", "subcategory"])["correct"]
    .agg(total="count", correct="sum")
    .assign(
        accuracy=lambda d: d["correct"] / d["total"],
        error_count=lambda d: d["total"] - d["correct"],
        error_rate=lambda d: 1 - d["correct"] / d["total"],
    )
    .sort_values("error_count", ascending=False)
    .head(TOP_N)
)
by_subcategory

In [ ]:
labels = [f"{cat} / {sub}" for cat, sub in by_subcategory.index]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(labels[::-1], by_subcategory["error_count"].values[::-1], color="steelblue")
ax.set_xlabel("Error count")
ax.set_title(f"Top {TOP_N} subcategories by error count")
plt.tight_layout()
plt.show()

## Derive category distribution

Algorithm:
1. Discard categories with fewer than `MIN_QUESTIONS` samples (too noisy to estimate error rate reliably).
2. Weight remaining categories by their **error rate** (not error count, to avoid biasing toward large categories).
3. Normalise those weights to sum to `1 - OTHER_RATIO`.
4. Set `other: OTHER_RATIO` — this distributes the remaining weight uniformly across all categories
   not listed explicitly (including the discarded low-sample ones) during dataset building.

Tune `MIN_QUESTIONS`, `OTHER_RATIO`, then paste the printed yaml block into your config.

In [ ]:
MIN_QUESTIONS = 20  # discard categories with fewer samples than this
OTHER_RATIO = (
    0.3  # fraction reserved for 'other' (randomly sampled from remaining categories)
)

eligible = by_category[by_category["total"] >= MIN_QUESTIONS].copy()

if eligible.empty:
    print(
        f"WARNING: no categories have >= {MIN_QUESTIONS} questions. Falling back to other: 1.0"
    )
    print("category_distribution:")
    print("  other: 1.0")
else:
    discarded = by_category[by_category["total"] < MIN_QUESTIONS]
    if not discarded.empty:
        print(
            f"Discarded {len(discarded)} category/ies with < {MIN_QUESTIONS} questions: {list(discarded.index)}\n"
        )

    raw_weights = eligible["error_rate"]
    explicit_total = 1 - OTHER_RATIO
    distribution = (
        (raw_weights / raw_weights.sum() * explicit_total)
        .sort_values(ascending=False)
        .round(4)
    )

    print("category_distribution:")
    for cat, w in distribution.items():
        print(f"  {cat}: {w}")
    print(f"  other: {OTHER_RATIO}")